In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

from v1sh_model.inputs.visualize import visualize_input, visualize_output
from v1sh_model.models.V1_model_2 import V1_model_2 as V1_model
from v1sh_model.inputs.examples import bar_without_surround

In [ ]:
# Instantiate and test the FullModel
seed = 42
model = V1_model(seed=seed)
T = 12
dt = 0.05
I_input = 1.5
N_y = 60
N_x = 100

In [ ]:
def stimulus(N_row = None, N_col = None, N_x = 4, N_y = 2, I_input = 1.5, orientation = 0.0):
    if N_row is None:
        N_row = N_y
    if N_col is None:
        N_col = N_x
    
    assert N_x % 2 == 0, "N_x must be even"
    middle_x = int(N_x / 2)
    A = np.full((N_y, N_x), orientation)
    A[:, middle_x:] = orientation + np.pi / 2
    
    C = np.zeros((N_y, N_x))
    assert N_y % 2 == 0, "N_y must be even"
    middle_y = int(N_y / 2)
    C[middle_y - N_row // 2 : middle_y + N_row // 2 + 1, middle_x - N_col // 2 : middle_x + N_col // 2 + 1] = I_input
    return A, C

In [ ]:
N_y = 60
N_x = 200
A, C = stimulus(N_row = 2, N_col = 48, N_x=N_x, N_y=N_y, I_input=I_input, orientation = 0.0000)
visualize_input(A, C, dpi = 400)
plt.show()

In [ ]:
X_gen, Y_gen, I = model.simulate(A, C, dt=dt, T=T, verbose=False, noisy=False, mode="wrap")

In [ ]:
model_output = model.g_x(X_gen).mean(axis=0) # N_y x N_x x K
C_out_half = model_output # .max(axis=-1)  # N_y x N_x
argmax_angle_indices = model_output.argmax(axis=-1)  # N_y x N_x
A_out_half = np.broadcast_to(model.M, (N_y, N_x, model.K))
x_middle = int(N_x / 2)
y_middle = int(N_y / 2)
visualize_output(A_out_half[24:-24, x_middle - 10:x_middle + 10], C_out_half[24:-24, x_middle - 10:x_middle + 10]* 3, verbose=False, dpi = 500)
plt.show()

In [ ]:
plt.figure(figsize=(6, 4), dpi=400)
plt.plot(model.g_x(X_gen)[:, 30, -20, 6], label="middle bar")
plt.plot(model.g_x(X_gen)[:, 30, 51, 6], label = "close to end bar")
plt.plot(model.g_x(X_gen)[:, 30, 50, 6], label = "end bar")
plt.plot(model.g_x(X_gen)[:, 30, 48:50, 6], label = "leaky neighbors")
plt.legend(loc = "upper right", fontsize = "small")
plt.xlabel("Time step")
plt.ylabel("Model response")
plt.tight_layout()
plt.show()

print(np.min(np.nonzero(C_half[30, :])[0]))  # first non-zero index